# 07 · Sparkplug B

**Objetivo:** entender o que o Sparkplug B acrescenta ao MQTT: namespace fixo, **certificados de nascimento e morte**, **aliases**, **números de sequência** e payload **protobuf** tipado.

| MQTT puro deixa em aberto | Sparkplug B define |
|---|---|
| estrutura de tópicos | `spBv1.0/<grupo>/<tipo>/<nó>[/<dispositivo>]` |
| formato do payload | protobuf com tipo de dado por métrica |
| como saber o que existe | NBIRTH / DBIRTH listam todas as métricas |
| como saber que morreu | NDEATH via Last Will + `bdSeq` |
| como saber que perdeu algo | `seq` de 0 a 255 |

### Antes de começar

| | |
|---|---|
| **Para quem** | Quem já montou tópicos MQTT próprios e quer entender o que uma especificação padronizada acrescenta. |
| **Pré-requisitos** | Notebooks 05 e 06 e um broker MQTT em `localhost:1883`. |
| **Tempo** | 30 minutos |

**Ao final você consegue:**

- Ler um tópico Sparkplug e identificar grupo, tipo de mensagem, nó e dispositivo.
- Explicar a sequência NBIRTH, DBIRTH, DDATA e NDEATH.
- Dizer para que servem `seq`, `bdSeq` e os aliases.
- Comparar o tamanho de um payload protobuf com o JSON equivalente.

**Onde isso aparece na fábrica:** SCADAs como o Ignition e vários gateways de borda falam Sparkplug B nativamente. Ele aparece quando o cliente exige que o SCADA descubra tags automaticamente e saiba quando um dado está obsoleto.

**Uma analogia para guardar:** MQTT puro é uma conversa em que cada um fala do seu jeito. Sparkplug é um formulário padrão: ao chegar, o dispositivo preenche a ficha de nascimento com tudo o que sabe medir, depois só informa o que mudou, e deixa registrado o aviso de falecimento.

> **Como estudar:** rode uma célula por vez. Sempre que aparecer **Pare e preveja**, responda mentalmente antes de executar. Errar a previsão é o que faz o conceito fixar.

In [1]:
# Configuração: deixa a raiz do repositório no caminho de import
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from common.notebook import load, run_in_background, require_broker
require_broker()
sp = load("07-sparkplug-b/sparkplug_demo.py")

broker MQTT encontrado em localhost:1883


## 1. Rodando o nó de borda e a aplicação host

O roteiro: o nó nasce (NBIRTH + DBIRTH), manda 6 DDATA, **perde uma mensagem de propósito**, manda mais uma, e morre sem desconectar.

> **Pare e preveja.** Antes de qualquer dado, o nó de borda publica um **certificado de nascimento**. O que você acha que precisa estar nele para o SCADA entender as mensagens seguintes?

In [2]:
resultado = sp.main()

Host application log:
  NBIRTH  edge-gw-01             seq=0    64 bytes  bdSeq, Node Control/Rebirth
  DBIRTH  edge-gw-01/filler-01   seq=1   176 bytes  Process/Speed, Process/Running, Counters/Good, Counters/Reject, Info/Recipe
  DDATA   edge-gw-01/filler-01   seq=2    57 bytes  Process/Speed, Process/Running, Counters/Good
  DDATA   edge-gw-01/filler-01   seq=3    57 bytes  Process/Speed, Process/Running, Counters/Good
  DDATA   edge-gw-01/filler-01   seq=4    57 bytes  Process/Speed, Process/Running, Counters/Good
  DDATA   edge-gw-01/filler-01   seq=5    57 bytes  Process/Speed, Process/Running, Counters/Good
  DDATA   edge-gw-01/filler-01   seq=6    57 bytes  Process/Speed, Process/Running, Counters/Good
  DDATA   edge-gw-01/filler-01   seq=7    57 bytes  Process/Speed, Process/Running, Counters/Good
  seq gap! expected 8, got 9 -> request rebirth
  DDATA   edge-gw-01/filler-01   seq=9    24 bytes  Counters/Reject
  NDEATH  edge-gw-01             seq=-    27 bytes  bdSeq

Current

**Interpretação do log**

1. **NBIRTH** (seq 0): o nó anuncia que está vivo, com `bdSeq` (número da sessão) e a métrica de controle `Rebirth`.
2. **DBIRTH** (seq 1): o dispositivo `filler-01` declara **todas as métricas**, com nome, tipo e **alias**. É o maior payload, porque leva os nomes.
3. **DDATA**: só os valores que mudaram, identificados pelo **alias numérico**. Bem menor.
4. **seq gap!**: pulamos um número de sequência de propósito; a aplicação host percebeu e, num sistema real, pediria um *rebirth* para se ressincronizar.
5. **NDEATH**: publicado **pelo broker** (era o Last Will), quando o socket caiu. O host marca o nó como offline.

## 2. Olhando um payload por dentro

In [3]:
import sparkplug_b_pb2 as spb
p = spb.Payload(seq=5)
sp.metric(p, alias=1, datatype=sp.FLOAT, value=417.5)
sp.metric(p, alias=3, datatype=sp.INT32, value=42)
bruto = p.SerializeToString()
print(f"{len(bruto)} bytes:", bruto.hex(" "))
print()
print(spb.Payload.FromString(bruto))

35 bytes: 12 10 10 01 18 96 bb c9 89 9d 34 20 09 65 00 c0 d0 43 12 0d 10 03 18 96 bb c9 89 9d 34 20 03 50 2a 18 05

metrics {
  alias: 1
  timestamp: 1794511101334
  datatype: 9
  float_value: 417.5
}
metrics {
  alias: 3
  timestamp: 1794511101334
  datatype: 3
  int_value: 42
}
seq: 5



> **Pare e preveja.** A mesma informação (duas métricas, timestamp e sequência) em protobuf com alias e em JSON. Qual é menor, e mais ou menos em que proporção?

In [4]:
import json
como_json = json.dumps({"Process/Speed": 417.5, "Counters/Good": 42, "timestamp": sp.now_ms(), "seq": 5})
print(f"protobuf com alias: {len(bruto)} bytes")
print(f"JSON equivalente  : {len(como_json)} bytes  ->  {como_json}")

protobuf com alias: 35 bytes
JSON equivalente  : 83 bytes  ->  {"Process/Speed": 417.5, "Counters/Good": 42, "timestamp": 1794511101339, "seq": 5}


**Interpretação:** o protobuf não manda nomes de campo, só números de campo e valores binários; o alias substitui o nome da métrica. Em links celulares pagos por megabyte, ou com milhares de tags, essa diferença é o motivo do Sparkplug existir.

## Resumo

* Sparkplug B = MQTT + **contrato**: qualquer host (Ignition, historiador, MES) entende qualquer nó sem configuração manual de tags.
* **Birth** dá o dicionário (nome, tipo, alias); **Data** manda só o que mudou; **Death** avisa a queda.
* `seq` e `bdSeq` permitem detectar perda e sessão trocada.
* Custo: payload binário (precisa do `.proto`) e um modelo mais rígido que o MQTT livre.

---
## Erros comuns

- **Publicar DATA sem ter publicado BIRTH.** A aplicação host não conhece os aliases nem os tipos e descarta a mensagem.
- **Não reiniciar `seq` no BIRTH.** O host usa o `seq` para detectar perda. Sequência errada dispara pedidos de rebirth em laço.
- **Tratar Sparkplug como substituto do UNS.** O namespace Sparkplug é fixo e orientado a dispositivo. Muitas arquiteturas espelham os dados para um UNS em ISA-95.
- **Ignorar o NDEATH.** Sem tratar a morte do nó, o SCADA continua mostrando o último valor como se fosse bom.

## Teste seu entendimento

Responda antes de abrir cada resposta.

**1. Para que serve o `bdSeq`?**

<details><summary>Ver resposta</summary>

Para casar um NDEATH com o NBIRTH da mesma sessão. Assim o host ignora um certificado de morte atrasado de uma conexão antiga.

</details>

**2. Por que as mensagens DDATA são tão menores que o DBIRTH?**

<details><summary>Ver resposta</summary>

O BIRTH carrega nome, tipo e alias de cada métrica. O DATA carrega só o alias numérico e o valor do que mudou.

</details>

**3. O que o host faz quando percebe um salto no `seq`?**

<details><summary>Ver resposta</summary>

Assume que perdeu uma mensagem e envia um comando de rebirth, pedindo que o nó publique de novo o estado completo.

</details>

## Exercícios para praticar

1. Adicione uma métrica nova ao dispositivo e veja o tamanho do DBIRTH crescer enquanto o DDATA quase não muda.
2. Derrube o nó de borda de forma abrupta e confirme que o host recebe o NDEATH publicado pelo broker.
3. Decodifique na mão os primeiros bytes do payload usando a explicação do protobuf: campo, tipo e valor.

## Glossário

| Termo | Significado |
|---|---|
| **NBIRTH / DBIRTH** | Certificados de nascimento do nó e do dispositivo, com a lista completa de métricas. |
| **NDATA / DDATA** | Mensagens de dados, só com o que mudou. |
| **NDEATH** | Certificado de morte do nó, registrado como Last Will. |
| **seq** | Contador de 0 a 255 que permite detectar mensagem perdida. |
| **Alias** | Número que substitui o nome da métrica nas mensagens de dados. |
| **Protobuf** | Formato binário compacto usado no payload Sparkplug B. |

**Próximo passo:** [08 · Gateway Modbus para MQTT](../08-modbus-to-mqtt-gateway/08_estudo_gateway_modbus_mqtt.ipynb), que junta o primeiro e o quinto notebooks num gateway de borda.